# Lightweight Fine-Tuning Project

TODO: In this cell, describe your choices for each of the following

* PEFT technique: LoRA
    - I was unsure, so I used the recommended PEFT technique, which was LoRA. It is also the PEFT technique that is compatible with all models, which makes it easier for me to pick any model I wanted.
* Model: bigscience/bloom-560m
    - bloom was pretrained on 45 natural languages making it already efffective at linguistic recognition, and 560m parameters is small enough for standard hardware and this workspace
* Evaluation approach: evaluate with a Hugging Face Trainer
    - this is the evaluation approach that was covered in the course, reliable
* Fine-tuning dataset: papluca/language-identification
    - large dataset 90k samples containing text in 20 different languages, and can help train my model in multi-class text classification

## Loading and Evaluating a Foundation Model

TODO: In the cells below, load your chosen pre-trained Hugging Face model and evaluate its performance prior to fine-tuning. This step includes loading an appropriate tokenizer and dataset.

In [1]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from datasets import load_dataset

In [2]:
bloom = "bigscience/bloom-560m"

In [3]:
# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(bloom)

In [4]:
# load model
model = AutoModelForSequenceClassification.from_pretrained(bloom, num_labels = 20)

Some weights of BloomForSequenceClassification were not initialized from the model checkpoint at bigscience/bloom-560m and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
raw_datasets = load_dataset("papluca/language-identification")

In [6]:
# create tokenized_dataset
from transformers import DataCollatorWithPadding

raw_datasets = raw_datasets.class_encode_column("labels")

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_datasets = raw_datasets.map(
    tokenize_function, 
    batched=True, 
    remove_columns=["text"]
)

def flatten_labels(example):
    if isinstance(example["labels"], list):
        example["labels"] = example["labels"][0]
    return example

tokenized_datasets = tokenized_datasets.map(flatten_labels)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tokenized_datasets["train"][0])

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

{'labels': 12, 'input_ids': [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,

In [7]:
from transformers import Trainer, DataCollatorWithPadding

# original model

original_model = AutoModelForSequenceClassification.from_pretrained(bloom, num_labels=20).to("cuda")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

original_trainer = Trainer(
    model=original_model,
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

# results before fine-tuning

original_results = original_trainer.evaluate()
print(f"Results Before Fine-Tuning: {original_results}")

Some weights of BloomForSequenceClassification were not initialized from the model checkpoint at bigscience/bloom-560m and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
You're using a BloomTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Results Before Fine-Tuning: {'eval_loss': 19.294605255126953, 'eval_runtime': 1115.7915, 'eval_samples_per_second': 8.962, 'eval_steps_per_second': 1.12}


## Performing Parameter-Efficient Fine-Tuning

TODO: In the cells below, create a PEFT model from your loaded model, run a training loop, and save the PEFT model weights.

In [8]:
# converting transformers model into peft model
from peft import get_peft_model
from peft import LoraConfig

config = LoraConfig(
    task_type="SEQ_CLS",
    r=8,                       
    lora_alpha=32,            
    target_modules=["query_key_value"], 
    lora_dropout=0.1,  
    bias="none"
)
lora_model = get_peft_model(model, config)
lora_model.print_trainable_parameters()

trainable params: 827,392 || all params: 560,041,984 || trainable%: 0.14773749533749242


In [10]:
# run training loop
from transformers import TrainingArguments, Trainer

raw_datasets_2 = load_dataset("papluca/language-identification")

raw_datasets_2 = raw_datasets_2.class_encode_column("labels")

tokenized_datasets_2 = raw_datasets_2.map(
    tokenize_function, 
    batched=True, 
    remove_columns=["text"]
)

def flatten_labels(example):
    example["labels"] = int(example["labels"])
    return example

tokenized_datasets_2 = tokenized_datasets_2.map(flatten_labels)

tokenized_datasets_2.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tokenized_datasets_2["train"][0])

training_args_2 = TrainingArguments(
    remove_unused_columns=False,
    per_device_train_batch_size = 8,
    fp16=True,  
    output_dir = "/tmp/training_results",
    learning_rate = 2e-5,
    num_train_epochs = 1,
    evaluation_strategy="epoch", 
    save_strategy="epoch",
)

trainer_2 = Trainer(
    model = lora_model,
    args = training_args_2,
    train_dataset = tokenized_datasets_2["train"],
    eval_dataset = tokenized_datasets_2["validation"],
    data_collator = data_collator,
    tokenizer = tokenizer,
)

trainer_2.train()

Map:   0%|          | 0/70000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

{'labels': tensor(12), 'input_ids': tensor([     3,      3,      3,      3,      3,      3,      3,      3,      3,
             3,      3,      3,      3,      3,      3,      3,      3,      3,
             3,      3,      3,      3,      3,      3,      3,      3,      3,
             3,      3,      3,      3,      3,      3,      3,      3,      3,
             3,      3,      3,      3,      3,      3,      3,      3,      3,
             3,      3,      3,      3,      3,      3,      3,      3,      3,
             3,      3,      3,      3,      3,      3,      3,      3,      3,
             3,      3,      3,      3,      3,      3,      3,      3,      3,
             3,      3,      3,      3,      3,      3,      3,      3,      3,
             3,      3,      3,      3,      3,      3,      3,      3,      3,
             3,      3,      3,      3,      3,      3,      3,      3,      3,
             3,      3,      3,      3,      3,      3,      3,      3,      3,
    

Epoch,Training Loss,Validation Loss
1,0.055900,0.158753


TrainOutput(global_step=8750, training_loss=0.5321990238734654, metrics={'train_runtime': 9349.685, 'train_samples_per_second': 7.487, 'train_steps_per_second': 0.936, 'total_flos': 6.518741139456e+16, 'train_loss': 0.5321990238734654, 'epoch': 1.0})

###  ⚠️ IMPORTANT ⚠️

Due to workspace storage constraints, you should not store the model weights in the same directory but rather use `/tmp` to avoid workspace crashes which are irrecoverable.
Ensure you save it in /tmp always.

In [11]:
# Saving the model
import os

lora_model.save_pretrained("/tmp/my_lora_model")
tokenizer.save_pretrained("/tmp/my_lora_model")

print("Final Directory Contents:", os.listdir("/tmp/my_lora_model"))

Final Directory Contents: ['adapter_model.bin', 'tokenizer_config.json', 'adapter_config.json', 'README.md', 'tokenizer.json', 'special_tokens_map.json']


## Performing Inference with a PEFT Model

TODO: In the cells below, load the saved PEFT model weights and evaluate the performance of the trained PEFT model. Be sure to compare the results to the results from prior to fine-tuning.

In [12]:
# load saved PEFT model

from peft import PeftModel
from transformers import BloomForSequenceClassification

base_model = BloomForSequenceClassification.from_pretrained(
    "bigscience/bloom-560m",
    num_labels=20
)

from peft import LoraConfig, PeftModel

# Manually load the config first
config = LoraConfig.from_pretrained("/tmp/my_lora_model")

# Force the task type if it was missing in your previous printout
if config.task_type is None:
    config.task_type = "SEQ_CLS"

# Load using the explicit config
peft_model = PeftModel.from_pretrained(base_model, "/tmp/my_lora_model", config=config).to("cuda")

#peft_model = PeftModel.from_pretrained(base_model, "/tmp/my_lora_model").to("cuda")

trainer_peft = Trainer(
    model=peft_model, 
    eval_dataset=tokenized_datasets["validation"], 
    data_collator=data_collator
)

# results after fine-tuning

fine_tuned_results = trainer_peft.evaluate()
print(f"Results After Fine-Tuning: {fine_tuned_results}")

Some weights of BloomForSequenceClassification were not initialized from the model checkpoint at bigscience/bloom-560m and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Results After Fine-Tuning: {'eval_loss': 0.15875273942947388, 'eval_runtime': 503.173, 'eval_samples_per_second': 19.874, 'eval_steps_per_second': 2.484}


In [13]:
improvement = original_results['eval_loss'] - fine_tuned_results['eval_loss']
print(f"Loss Reduction: {improvement}")

if improvement > 0:
    print("Fine-tuning successful! The model has learned from your data.")
else:
    print("No improvement detected. You might need more epochs or a higher learning rate.")

Loss Reduction: 19.13585251569748
Fine-tuning successful! The model has learned from your data.


In [14]:
#After fine-tuning, the model had a loss reduction of 19.1, implying a significant improvement.
#Results Before Fine-Tuning: {'eval_loss': 19.294605255126953 ...
#Results After Fine-Tuning: {'eval_loss': 0.15875273942947388 ...
#Loss Reduction: 19.13585251569748
#Fine-tuning successful! The model has learned from your data.
#The low eval loss implies high accuracy for the model after fine-tuning.